In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

import smtplib
from email.mime.text import MIMEText

sns.set_style("whitegrid")

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df.drop('customerID', axis=1, inplace=True)

emails = [
    "ANY_EMAIL@gmail.com",
    "ANY_EMAIL@gmail.com",
    "ANY_EMAIL@gmail.com"
]

df['Email'] = [emails[i % len(emails)] for i in range(len(df))]

le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object' and col != 'Email':
        df[col] = le.fit_transform(df[col])

X = df.drop(['Churn', 'Email'], axis=1)
y = df['Churn']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

df['Churn_Probability'] = model.predict_proba(X_scaled)[:, 1]

df['Risk_Level'] = pd.cut(
    df['Churn_Probability'],
    bins=[0, 0.3, 0.7, 1],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

high_risk_df = df[df['Risk_Level'] == 'High Risk']

def send_email(receiver_email, tenure, charges):
    sender_email = "YOUR_EMAIL@gmail.com"
    password = "YOUR_APP_PASSWORD"

    subject = "Customer Churn Risk Alert"

    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; line-height:1.6;">

        <h2 style="color:#d9534f;">Customer Churn Risk Alert</h2>

        <p>Dear Customer,</p>

        <p>
        This is an automated notification generated as part of a data analytics project.
        Based on behavioral patterns, the model has identified a higher likelihood of churn.
        </p>

        <p><b>Customer Summary:</b></p>
        <ul>
            <li><b>Tenure:</b> {tenure} months</li>
            <li><b>Monthly Charges:</b> ₹{charges}</li>
        </ul>

        <p>
        This insight can help in taking proactive steps such as improving user experience
        or offering personalized plans.
        </p>

        <p style="font-size:12px; color:gray;">
        Note: This message is generated for demonstration purposes only.
        </p>

        <br>

        <p>Regards,<br>
        <b>Abhijith</b><br>
        Data Analytics Project</p>

    </body>
    </html>
    """

    msg = MIMEText(html_body, "html")
    msg['Subject'] = subject
    msg['From'] = sender_email
    msg['To'] = receiver_email

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        server.quit()
        print(f"Email sent to {receiver_email}")
    except Exception as e:
        print(f"Failed for {receiver_email}: {e}")

for i, row in high_risk_df.head(5).iterrows():
    send_email(row['Email'], row['tenure'], row['MonthlyCharges'])

/tmp/ipykernel_8871/2967813880.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


✅ Email sent to 2320030006@klh.edu.in
✅ Email sent to 2320030006@klh.edu.in
✅ Email sent to 2320030004cse@gmail.com
✅ Email sent to 2320030006cse@gmail.com
✅ Email sent to 2320030006@klh.edu.in
